# Fine-tune and generate
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1/cookbook/notebooks/05_finetune_and_generate.ipynb)

Adapt the 20M model to a sequence set, save and reload it, and compare new samples with base-model samples and training references.

This notebook runs independently. Select **Runtime → Change runtime type → GPU** in Colab.
First use downloads model weights. Training is a small workflow demonstration, not a converged design experiment. GPU memory requirements depend on the model, batch size, and length; a free Colab GPU is not guaranteed to fit RL-SAE.
The setup installs the `v1` release when IDiom is absent. If using an older installation,
upgrade to that release and restart the kernel. No adjacent helper files are required.

In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "idiom[cookbook] @ git+https://github.com/rotskoff-group/idiom.git@v1"])
if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas>=2"])

import json
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from idiom import IDiom, IDiomSAE
from idiom.data.records import Record
from idiom.utils.notebook_helpers import (
    load_inputs, idr_sequence, isolated, check_context, summaries, write_fasta,
    save_run, sequence_metrics, nearest_reference, split_records, example_file,
)
print("Python:", sys.version.split()[0])
started = time.perf_counter()

## Inputs and settings
Run top to bottom. Upload a FASTA using Colab's Files pane and set its path below, or
leave the demo input unchanged. `INPUT_MODE="idr"` accepts ordinary headers for isolated
IDRs; `"annotated"` requires full-protein headers ending in `_IDR_x-y` (1-based inclusive).
Python coordinates are 0-based, end-exclusive. These workflows do not predict IDR boundaries.

For persistent outputs, optionally mount Drive in your own cell with
`from google.colab import drive; drive.mount("/content/drive")`, then set `OUT_DIR` there.
Use a new output directory for each experiment. Rejected records are reported in an audit.

In [ ]:
INPUT_FASTA = None # Default: bundled nucleolus IDRs
INPUT_MODE = "idr"
MAX_RECORDS = 32
MODEL_ID = "jxliu2/idiom-20M"
DEVICE = "auto"
BATCH_SIZE = 2
SEED = 0
MAX_STEPS = 20
LEARNING_RATE = 1e-5
VALIDATION_FRACTION = 0.2
RESUME_FROM = None # Path to training/checkpoints/last.ckpt to resume toward MAX_STEPS
N = 8
MAX_NEW_TOKENS = 96
OUT_DIR = Path("sft_outputs")

In [ ]:
import gc
import torch
import lightning as L
from importlib.resources import files
from omegaconf import OmegaConf
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
L.seed_everything(SEED, workers=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
if (OUT_DIR / "training").exists() and RESUME_FROM is None:
    raise ValueError("Use a fresh OUT_DIR or set RESUME_FROM to a training checkpoint.")
from idiom.utils.device import resolve_device
training_device = resolve_device(DEVICE)
precision = "bf16-mixed" if training_device.type == "cuda" and torch.cuda.is_bf16_supported() else "32-true"
print("CUDA:", torch.cuda.is_available(), "Training precision:", precision)

## Prepare training and held-out sequences
Exact duplicate IDRs stay in the same split. This is not a homology-aware split: use externally
clustered data for rigorous generalization studies. The demo fine-tunes on isolated IDRs using
completion-only loss. Original flanks are not used. Oversized sequences are reported and excluded.

In [ ]:
input_path = INPUT_FASTA or example_file("protgps/nucleolus.fasta", OUT_DIR / "inputs")
records, audit = load_inputs(input_path, INPUT_MODE, MAX_RECORDS)
display(audit)
if not records:
    raise ValueError("No accepted sequences.")

## Sample the base model
Use identical generation settings before and after training.

In [ ]:
base = IDiom.from_pretrained(MODEL_ID, device=DEVICE)
sampling = dict(n=N, batch_size=BATCH_SIZE, seed=SEED, temperature=1.0,
                max_new_tokens=MAX_NEW_TOKENS)
baseline = base.generate_unprompted(**sampling)
write_fasta([Record(f"baseline_{i}", s, 0, len(s)) for i, s in enumerate(baseline) if s],
            OUT_DIR / "baseline.fasta")
model_context = base.model.cfg.max_seq_len
del base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
too_long = [r.accession for r in records if len(idr_sequence(r)) + 4 > model_context]
audit.loc[audit.record_id.isin(too_long), "status"] = "exceeds model context"
records = isolated([r for r in records if r.accession not in too_long])
train_records, validation_records = split_records(records, validation_fraction=VALIDATION_FRACTION, seed=SEED)
audit["split"] = audit.record_id.map({**{r.accession: "train" for r in train_records},
                                      **{r.accession: "validation" for r in validation_records}})
audit.to_csv(OUT_DIR / "input_audit.csv", index=False)
train_path = write_fasta(train_records, OUT_DIR / "train.fasta")
validation_path = write_fasta(validation_records, OUT_DIR / "validation.fasta")
print("Train:", len(train_records), "Validation:", len(validation_records))

## Fine-tune
The packaged training builder supplies the same model and data logic as the SFT CLI.
We use local CSV logs, so no tracking account is needed. Validation loss is recorded during
training; a short run may not improve it. Increase steps only after inspecting the split and logs.

In [ ]:
from idiom.train.autoreg.train_autoreg import build
cfg = OmegaConf.load(files("idiom") / "configs/sft.yaml")
cfg.init_from = MODEL_ID
cfg.seed = SEED
cfg.data.train_fasta = str(train_path.resolve())
cfg.data.val_fasta = str(validation_path.resolve())
cfg.data.prompted_prob = 0.0
cfg.data.batch_size = min(BATCH_SIZE, len(train_records))
cfg.data.num_workers = 0
cfg.optim.lr = LEARNING_RATE
cfg.optim.warmup_steps = min(5, MAX_STEPS)
cfg.trainer.max_steps = MAX_STEPS
cfg.out_dir = str(OUT_DIR.resolve())
cfg.resume_from = RESUME_FROM
cfg.device = str(training_device)
cfg.trainer = dict(max_steps=MAX_STEPS, accelerator="gpu" if training_device.type == "cuda" else "cpu",
                   devices=[training_device.index or 0] if training_device.type == "cuda" else 1,
                   precision=precision, gradient_clip_val=1.0, accumulate_grad_batches=1,
                   log_every_n_steps=1, limit_val_batches=2, num_sanity_val_steps=0,
                   enable_model_summary=False)
OmegaConf.save(cfg, OUT_DIR / "training_config.yaml")
lit, dm = build(cfg)

In [ ]:
checkpoint = ModelCheckpoint(dirpath=OUT_DIR / "training/checkpoints", save_last=True,
                             save_top_k=0, every_n_train_steps=max(1, min(10, MAX_STEPS)))
trainer = L.Trainer(**OmegaConf.to_container(cfg.trainer, resolve=True),
                    logger=CSVLogger(OUT_DIR / "training", name="metrics"), callbacks=[checkpoint])
trainer.fit(lit, datamodule=dm, ckpt_path=RESUME_FROM)
validation_metrics = trainer.validate(lit, datamodule=dm)
(OUT_DIR / "validation_metrics.json").write_text(json.dumps(validation_metrics, indent=2))

## Save, reload, and generate

In [ ]:
trainer.save_checkpoint(OUT_DIR / "training/checkpoints/last.ckpt")
release = OUT_DIR / "model"
IDiom(lit.model.cpu()).save_pretrained(release)
del trainer, lit
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
adapted_model = IDiom.from_pretrained(release, device=DEVICE)
adapted = adapted_model.generate_unprompted(**sampling)
write_fasta([Record(f"adapted_{i}", s, 0, len(s)) for i, s in enumerate(adapted) if s],
            OUT_DIR / "adapted.fasta")
del adapted_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Compare samples and check similarity to training data
SequenceMatcher similarity below is a string-comparison diagnostic, not alignment identity.
Exact training matches are reported separately. A short run is not evidence of generalization.

In [ ]:
comparison = pd.concat([sequence_metrics(baseline).assign(group="baseline"),
                        sequence_metrics(adapted).assign(group="adapted")], ignore_index=True)
comparison.to_csv(OUT_DIR / "candidates.csv", index=False)
display(comparison.groupby("group").agg(count=("sequence", "size"), mean_length=("length", "mean"),
                                        mean_entropy=("entropy", "mean"), duplicate_fraction=("duplicate", "mean")))
fig, axes = plt.subplots(1, 2, figsize=(8, 3), constrained_layout=True)
for group, rows in comparison.groupby("group"):
    axes[0].hist(rows.length, bins=10, alpha=0.5, label=group)
    axes[1].hist(rows.entropy, bins=10, alpha=0.5, label=group)
axes[0].set(xlabel="Length", ylabel="Count")
axes[1].set(xlabel="Composition entropy (bits)")
axes[0].legend()
fig.savefig(OUT_DIR / "comparison.png", dpi=160)
plt.show()

nearest = nearest_reference(adapted, [idr_sequence(r) for r in train_records])
nearest.to_csv(OUT_DIR / "nearest_training_sequences.csv", index=False)
display(nearest)
save_run(OUT_DIR, dict(model=MODEL_ID, seed=SEED, steps=MAX_STEPS, learning_rate=LEARNING_RATE,
                       sampling=sampling, input=str(input_path), resume_from=RESUME_FROM),
         elapsed=time.perf_counter() - started)

## Save and continue
`model/` is a reloadable release; `training/checkpoints/last.ckpt` also preserves optimizer
state for resuming. `training/metrics/` contains CSV training logs. Keep the training config,
input audit, and run settings with generated sequences. These short runs demonstrate the workflow;
assess held-out data, diversity, and independent measurements before drawing design conclusions.
The download includes checkpoints and can be large; use Drive for long-running experiments.

Next: [custom rewards](06_design_with_custom_rewards.ipynb).

In [ ]:
import shutil
archive = shutil.make_archive(str(OUT_DIR.resolve()), "zip", OUT_DIR)
print("Results:", OUT_DIR.resolve(), "\nDownload:", archive)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(archive)